In [ ]:
# --- Cell 0.1: Залежності ---
!pip install -q "sentence-transformers>=3.0" qdrant-client pandas numpy "transformers>=4.41.0"

# --- Cell 0.2: Імпорти ---
import os
import glob
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from google.colab import drive, userdata

drive.mount("/content/drive")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- Cell 0.3: Конфіг ---
CFG = {
    # Моделі на Drive
    "embedder_path": "/content/drive/MyDrive/qwen-cv-retriever-finetuned-v2",
    "reranker_path": "/content/drive/MyDrive/bge-reranker-cv-finetuned-distilled",

    # Qdrant колекції (з ембеддінгами від fine-tuned моделі)
    "collection_cvs": "trained_embedder_cvs",
    "collection_jobs": "trained_embedder_jobs",

    # Pipeline
    "retrieval_limit": 30,      # скільки дістати з Qdrant
    "final_top_k": 10,          # скільки лишити після rerank
    "max_seq_length": 512,      # для reranker
    "max_text_length": 2000,    # обрізка довгих текстів

    # Промпти embedder (мають збігатись з train-часом!)
    "doc_prompt": "Represent this candidate CV for job matching retrieval",
    "query_prompt": "Find candidates matching this job vacancy",
}
print("Config готовий")

# --- Cell 0.4: Qdrant ---
client = QdrantClient(
    url=userdata.get("QDRANT_URL"),
    api_key=userdata.get("QDRANT_API_KEY"),
    timeout=300,
)

collections = [c.name for c in client.get_collections().collections]
print(f"Доступні колекції: {collections}")

for name in [CFG["collection_cvs"], CFG["collection_jobs"]]:
    if name in collections:
        info = client.get_collection(name)
        print(f"  {name}: {info.points_count} точок")
    else:
        print(f"  ⚠ {name} — НЕ ЗНАЙДЕНО")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 32.4 MB/s eta 0:00:00
Mounted at /content/drive
Device: cuda
GPU: NVIDIA L4
Config готовий
Доступні колекції: ['candidates', 'jobs', 'trained_embedder_cvs', 'trained_embedder_jobs']
  trained_embedder_cvs: 14933 точок
  trained_embedder_jobs: 8150 точок


In [ ]:
# --- Cell 1.1: Fine-tuned embedder ---
print("Завантаження embedder...")
embedder = SentenceTransformer(
    CFG["embedder_path"],
    model_kwargs={"torch_dtype": torch.float16} if device == "cuda" else {},
)
embedder.max_seq_length = 1024
print(f"✅ Embedder: {CFG['embedder_path']}")

# --- Cell 1.2: Fine-tuned reranker ---
print("Завантаження reranker...")
reranker = CrossEncoder(
    CFG["reranker_path"],
    max_length=CFG["max_seq_length"],
)
print(f"✅ Reranker: {CFG['reranker_path']}")


Завантаження embedder...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

✅ Embedder: /content/drive/MyDrive/qwen-cv-retriever-finetuned-v2
Завантаження reranker...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ Reranker: /content/drive/MyDrive/bge-reranker-cv-finetuned-distilled


In [ ]:
# --- Cell 2.1: Helper для тексту вакансії ---
def strip_instruct(text):
    """Прибирає Qwen-обгортку 'Instruct: ...\\nQuery: <текст>'."""
    text = str(text)
    if "Query:" in text:
        return text.split("Query:", 1)[1].strip()
    return text.strip()


def clip_text(text, n=None):
    n = n or CFG["max_text_length"]
    return str(text)[:n]


def build_vacancy_text_from_payload(payload):
    """
    Формує текст запиту з payload вакансії Qdrant.
    Використовує embed_text якщо є, інакше збирає з окремих полів.
    """
    if payload.get("embed_text"):
        return clip_text(strip_instruct(payload["embed_text"]))

    # Fallback
    parts = []
    if payload.get("title"):
        parts.append(f"Title: {payload['title']}")
    if payload.get("skills_tags"):
        parts.append(f"Skills: {payload['skills_tags']}")
    if payload.get("experience_text"):
        parts.append(f"Experience: {payload['experience_text']}")
    return clip_text("\n".join(parts))


# --- Cell 2.2: Dense retrieval через Qdrant ---
def dense_retrieve(vacancy_text, filters=None, limit=None):
    """
    Семантичний пошук: embed вакансію → пошук у Qdrant → top-N CV.
    """
    limit = limit or CFG["retrieval_limit"]

    query_vec = embedder.encode(
        vacancy_text,
        prompt=CFG["query_prompt"],
        normalize_embeddings=True,
    ).tolist()

    must = []
    if filters:
        for key, value in filters.items():
            must.append(FieldCondition(key=key, match=MatchValue(value=value)))
    query_filter = Filter(must=must) if must else None

    results = client.query_points(
        collection_name=CFG["collection_cvs"],
        query=query_vec,
        query_filter=query_filter,
        limit=limit,
        with_payload=True,
    )
    return results.points


# --- Cell 2.3: Reranking через cross-encoder ---
def rerank_candidates(vacancy_text, candidates, top_k=None):
    """
    Cross-encoder перераховує кандидатів, повертає top_k.
    """
    top_k = top_k or CFG["final_top_k"]

    if not candidates:
        return []

    pairs = [
        [vacancy_text, clip_text(hit.payload.get("cv_text", ""))]
        for hit in candidates
    ]
    scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)

    # Нормалізуємо cross-encoder scores до 0-1 (sigmoid уже применений у BCE-trained модели,
    # але тут scores можуть бути будь-які — беремо як є, порядок важливий)
    ranked = []
    for hit, ce_score in zip(candidates, scores):
        ranked.append({
            "candidate_id": hit.payload.get("candidate_id", str(hit.id)),
            "full_name": hit.payload.get("full_name", ""),
            "title": hit.payload.get("title", ""),
            "seniority": hit.payload.get("seniority", ""),
            "region": hit.payload.get("region", ""),
            "skills": hit.payload.get("skills", []),
            "linkedin_url": hit.payload.get("linkedin_url", ""),
            "cv_text": hit.payload.get("cv_text", ""),
            "dense_score": float(hit.score),
            "rerank_score": float(ce_score),
        })

    ranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return ranked[:top_k]


    # --- Cell 2.4: Повний pipeline ---
def rank_for_vacancy(vacancy_text, filters=None, top_k=None, verbose=True):
    """
    Головна функція. Приймає текст вакансії, повертає ранжованих кандидатів.

    Args:
        vacancy_text: рядок з описом вакансії
        filters: dict {payload_field: value} для фільтрації (seniority, region тощо)
        top_k: скільки кандидатів у фінальний список
        verbose: друкувати результати

    Returns:
        List[dict] ранжовані кандидати
    """
    # Етап 1: Dense retrieval
    candidates = dense_retrieve(vacancy_text, filters=filters)

    # Етап 2: Reranking
    ranked = rerank_candidates(vacancy_text, candidates, top_k=top_k)

    if verbose:
        print(f"Retrieved: {len(candidates)} → Reranked: {len(ranked)}\n")
        for i, r in enumerate(ranked, 1):
            print(f"{i}. {r['full_name']} — rerank={r['rerank_score']:.4f} | dense={r['dense_score']:.4f}")
            print(f"   {r['title']} | {r['seniority']} | {r['region']}")
            print(f"   Skills: {r['skills']}")
            print()

    return ranked


def rank_for_vacancy_by_id(job_id, filters=None, top_k=None, verbose=True):
    """Обгортка: тягне вакансію з Qdrant по job_id і запускає pipeline."""
    points = client.retrieve(
        collection_name=CFG["collection_jobs"],
        ids=[job_id],
        with_payload=True,
    )
    if not points:
        print(f"⚠ Вакансія {job_id} не знайдена")
        return []

    payload = points[0].payload
    vacancy_text = build_vacancy_text_from_payload(payload)

    if verbose:
        print(f"🔍 {payload.get('title', 'N/A')} @ {payload.get('company', 'N/A')}")
        print(f"   {payload.get('experience_text', '')}")
        print(f"   {payload.get('skills_tags', '')}")
        print("=" * 60)

    return rank_for_vacancy(vacancy_text, filters=filters, top_k=top_k, verbose=verbose)


print("Pipeline готовий")

Pipeline готовий


In [ ]:
# --- Cell 3.1: Список вакансій з Qdrant для вибору ---
def list_vacancies(limit=20, offset=None):
    """Показує доступні вакансії з Qdrant."""
    points, _ = client.scroll(
        collection_name=CFG["collection_jobs"],
        limit=limit,
        offset=offset,
        with_payload=True,
    )
    for i, p in enumerate(points):
        title = p.payload.get("title", "N/A")
        company = p.payload.get("company", "N/A")
        exp = p.payload.get("experience_text", "")
        print(f"[{i}] {title} — {company} | {exp}")
        print(f"    id: {p.id}")
    return points


# --- Cell 3.2: Приклад використання ---
# vacancies = list_vacancies(limit=10)
# results = rank_for_vacancy_by_id(vacancies[0].id, top_k=10)


# --- Cell 3.3: Кастомна вакансія (не з бази) ---
# custom_vacancy = """
# Title: Senior Python Backend Engineer
# Company: FinTech Startup
# Skills: Python, FastAPI, PostgreSQL, Docker, AWS
# Experience: 5+ years
#
# We're looking for an experienced Python backend developer...
# """
# results = rank_for_vacancy(custom_vacancy, top_k=10)


# --- Cell 3.4: З фільтрами ---
# results = rank_for_vacancy_by_id(
#     vacancies[0].id,
#     filters={"seniority": "Senior", "region": "Europe"},
#     top_k=5,
# )

In [ ]:
# --- Cell 4.1: Багато вакансій одразу → результати в DataFrame ---
def batch_rank(job_ids, top_k=10, filters=None):
    """
    Ранжує кандидатів для списку вакансій.
    Повертає DataFrame з (job_id, rank, candidate_id, scores...).
    """
    rows = []
    for job_id in tqdm(job_ids, desc="Batch ranking"):
        try:
            ranked = rank_for_vacancy_by_id(
                job_id, filters=filters, top_k=top_k, verbose=False
            )
            for rank, cand in enumerate(ranked, 1):
                rows.append({
                    "job_id": job_id,
                    "rank": rank,
                    "candidate_id": cand["candidate_id"],
                    "full_name": cand["full_name"],
                    "title": cand["title"],
                    "seniority": cand["seniority"],
                    "region": cand["region"],
                    "dense_score": cand["dense_score"],
                    "rerank_score": cand["rerank_score"],
                    "linkedin_url": cand["linkedin_url"],
                })
        except Exception as e:
            print(f"  ⚠ {job_id}: {e}")

    return pd.DataFrame(rows)


# --- Cell 4.2: Запис результатів на Drive ---
def save_batch_results(results_df, filename="ranking_results.csv"):
    path = f"/content/drive/MyDrive/{filename}"
    results_df.to_csv(path, index=False)
    print(f"Збережено → {path}")


# Приклад:
# vacancies_sample = list_vacancies(limit=20)
# job_ids = [v.id for v in vacancies_sample]
# results_df = batch_rank(job_ids, top_k=10)
# save_batch_results(results_df, "top10_per_vacancy.csv")
# print(results_df.head(20))

In [ ]:
# ============================================================================
#  SECTION 5 — (ОПЦІЙНО) EVALUATION НА GOLDEN
# ============================================================================
#
#  Швидка перевірка що pipeline дає ті самі метрики, що були в тренуванні.
#  Використовує ту саму папку GoldenV2, що і при тренуванні reranker.

# --- Cell 5.1: Завантажити golden ---
from sklearn.metrics import ndcg_score

GOLDEN_DIR = "/content/drive/MyDrive/CV_rank_Datasets/GoldenV2"


def load_golden_for_eval():
    """Читає golden CSV + тягне тексти вакансій з Qdrant."""
    _cache = {}

    def get_vacancy(job_id):
        if job_id in _cache:
            return _cache[job_id]
        points = client.retrieve(
            collection_name=CFG["collection_jobs"],
            ids=[job_id],
            with_payload=True,
        )
        text = build_vacancy_text_from_payload(points[0].payload) if points else ""
        _cache[job_id] = text
        return text

    files = sorted(glob.glob(f"{GOLDEN_DIR}/*eval_pool_*.csv"))
    print(f"Golden файлів: {len(files)}")

    golden = {}
    for path in tqdm(files, desc="Читання golden"):
        df = pd.read_csv(path)
        if "job_id" not in df.columns or "cv_text" not in df.columns:
            continue

        job_id = df["job_id"].iloc[0]
        vacancy_text = get_vacancy(job_id)
        if not vacancy_text:
            continue

        cands = [
            (clip_text(row["cv_text"]), int(row["Relevance_Score_0_to_3"]))
            for _, row in df.iterrows()
            if pd.notna(row.get("cv_text"))
        ]
        if cands:
            golden[job_id] = {"vacancy_text": vacancy_text, "candidates": cands}

    print(f"Валідних: {len(golden)}")
    return golden


# --- Cell 5.2: Full pipeline vs baseline (rerank on/off) ---
def evaluate_full_pipeline(golden, k=10, use_rerank=True):
    """Прогоняє повний pipeline на golden і рахує NDCG@k / MRR."""
    ndcgs, mrrs = [], []

    for job_id, data in tqdm(golden.items(), desc="Eval"):
        vacancy = data["vacancy_text"]
        cvs = [c[0] for c in data["candidates"]]
        labels = np.array([c[1] for c in data["candidates"]])

        if labels.max() == 0:
            continue

        if use_rerank:
            pairs = [[vacancy, cv] for cv in cvs]
            scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)
        else:
            # dense only
            q_vec = embedder.encode(
                vacancy, prompt=CFG["query_prompt"], normalize_embeddings=True
            )
            cv_vecs = embedder.encode(
                cvs, prompt=CFG["doc_prompt"], normalize_embeddings=True,
                batch_size=8, show_progress_bar=False,
            )
            scores = cv_vecs @ q_vec

        ndcgs.append(ndcg_score([labels], [scores], k=k))

        order = np.argsort(scores)[::-1]
        for rank, idx in enumerate(order, 1):
            if labels[idx] >= 2:
                mrrs.append(1.0 / rank)
                break
        else:
            mrrs.append(0.0)

    return {
        "ndcg@10": np.mean(ndcgs) if ndcgs else 0.0,
        "mrr": np.mean(mrrs) if mrrs else 0.0,
        "n": len(ndcgs),
    }


# Приклад:
# golden = load_golden_for_eval()
# dense_only = evaluate_full_pipeline(golden, use_rerank=False)
# full = evaluate_full_pipeline(golden, use_rerank=True)
#
# print(f"Dense only:  NDCG@10={dense_only['ndcg@10']:.4f} MRR={dense_only['mrr']:.4f}")
# print(f"Dense+Rerank: NDCG@10={full['ndcg@10']:.4f} MRR={full['mrr']:.4f}")

In [ ]:
# ============================================================================
#  QUICK START — типовий workflow
# ============================================================================
#
#  # Показати вакансії
vacancies = list_vacancies(limit=10)
#
#  # Ранжувати кандидатів для однієї
results = rank_for_vacancy_by_id(vacancies[1].id, top_k=10)
#
#  # З фільтром
results = rank_for_vacancy_by_id(
    vacancies[0].id,
    filters={"seniority": "Senior"},
    top_k=5,
)
#
#  # Батчем на 50 вакансій
job_ids = [v.id for v in list_vacancies(limit=50)]
df = batch_rank(job_ids, top_k=10)
save_batch_results(df)
#
#  # Перевірка якості на golden
golden = load_golden_for_eval()
metrics = evaluate_full_pipeline(golden)
print(metrics)
# ============================================================================

[0] Sitecore Developer — DevHandler | Only from 3 years of experience
    id: 0011e4cc-2ea1-4ddc-934b-b17448884940
[1] Media Buyer (Push Traffic) — BET Partner | Only from 2 years of experience
    id: 001235b8-f30d-46d8-b007-915d8876164e
[2] Embedded Software Engineer — ITExpert | Only from 5 years of experience
    id: 00270844-05e3-4cd2-88a9-5e01bc8bec1d
[3] Senior DevOps Engineer (AWS) — Zultys | Only from 6 years of experience
    id: 002d513f-864d-43ce-8e25-667f2cacbc8d
[4] Chief Marketing Officer - E-commerce — Visit Ukraine | Only from 4 years of experience
    id: 002de38c-c804-478f-b897-f0ee7228f3ab
[5] Fullstack Python LLM Developer — Uasoftdev | from 4 years of experience Considering with 3 years of experience
    id: 004727de-f1c7-4369-9c07-9704580ef3df
[6] Інженер з налаштування польотних контролерів БПЛА — Strix Air | Only from 1 year of experience
    id: 0069fb67-42b5-4200-ae00-ab900eca9458
[7] Data Science Engineer — MODUS X | from 4 years of experience Considering wi

UnexpectedResponse: Unexpected Response: 400 (Bad Request)
Raw response content:
b'{"status":{"error":"Bad request: Index required but not found for \\"seniority\\" of one of the following types: [keyword]. Help: Create an index for this key or use a different filter."},"time":0.00 ...'

In [ ]:
# Витягнемо повний cv_text для Aravindhan
for cand in results:
    if cand["full_name"] == "Ankush Kumar":
        print("=== FULL CV TEXT ===")
        print(cand["cv_text"][:2000])
        print("\n=== PAYLOAD FIELDS ===")
        print(f"Title: {cand['title']}")
        print(f"Skills: {cand['skills']}")
        print(f"Seniority: {cand['seniority']}")
        break

=== FULL CV TEXT ===
Seniority: Senior
Location: Sahibzada Ajit Singh Nagar, Punjab
Experience: 5-10 years

Summary: Social Media expert who understand the nuances of search engine optimization (SEO). Adept at identifying opportunities to improve Marketing and achieve better web traffic, leads and conversions. Specialise in analyzing a website's overall performance using Google Analytics to identify areas for improvement with a goal of increasing site traffic, conversions, and sales. A dedicated team player that enjoys helping clients gain recognition online by creating engaging digital content

=== PAYLOAD FIELDS ===
Title: Unknown
Skills: 
Seniority: Senior


In [ ]:
vacancies[1]

Record(id='001235b8-f30d-46d8-b007-915d8876164e', payload={'url': 'https://djinni.co/jobs/825145-media-buyer-push-traffic/', 'title': 'Media Buyer (Push Traffic)', 'company': 'BET Partner', 'category': 'Social Media', 'skills_tags': 'Marketing & Sales, Marketing, Digital Marketing, Social Media', 'experience_text': 'Only from 2 years of experience', 'experience_months': 24.0, 'employment_type': 'FULL_TIME', 'work_format': 'Full Remote', 'location': '', 'domain': 'Gambling', 'embed_text': 'Instruct: Given a job listing, retrieve the most relevant candidate profiles that match the required skills, experience, and role.\nQuery: Title: Media Buyer (Push Traffic)\nCompany: BET Partner\nCategory: Social Media\nSkills: Marketing & Sales, Marketing, Digital Marketing, Social Media\nLanguages: English - B2 - Upper Intermediate, Ukrainian - C1 - Advanced\nExperience: 2 years experience\nEmployment: FULL_TIME\nWork format: Full Remote\nLocation: \nDomain: Gambling\n\nDescription:\nПро компанію\nШ